<a href="https://colab.research.google.com/github/muhyassin09/yasinnn/blob/main/menuju-indonesia-emas-2045/notebooks/04_finalizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Name: Indonesian Municipal Fiscal Analytics Framework
## Module: 04_Finalizing

**Project Pipeline Status:**
- [x] **00_Data_Acquisition.ipynb** -> Documents the source publication and how the raw table data was extracted from BPS PDF reports.
- [x] **01_Data_Preprocessing.ipynb** -> Loads the raw BPS fiscal CSV and runs data-quality checks (shape, dtypes, missing values, duplicates).
- [x] **02_Data_Analysis.ipynb** -> Explores distribution shape/skewness of the 8 fiscal ratios and applies a log1p transform where it helps.
- [x] **03_Modelling.ipynb** -> Standardizes features, selects k, fits K-Means (k=4), and profiles/visualizes the resulting clusters.
- [ ] **04_Finalizing.ipynb** *(current)* -> Checks the geographic pattern of clusters, saves the final labeled dataset, and writes the project summary.

---
### 🎯 Module Objective
Sanity-check the K-Means clusters against real geography, save the final labeled dataset for downstream use, and summarize the end-to-end pipeline outcome and its explicit limitations.

### 📥 Data Ingestion
* **Source File:** `data/processed/fiscal_clustered_with_pca.csv`
* **Current Shape:** 508 rows × 22 columns (original + log + cluster + cluster_label + pc1/pc2)

---


### 🛠️ Environment Setup

In [4]:
import pandas as pd

In [1]:
!git clone https://github.com/muhyassin09/yasinnn/

Cloning into 'yasinnn'...
remote: Enumerating objects: 676, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 676 (delta 63), reused 9 (delta 9), pack-reused 583 (from 1)
Receiving objects: 100% (676/676), 19.58 MiB | 38.93 MiB/s, done.
Resolving deltas: 100% (299/299), done.


In [5]:
df = pd.read_csv('/content/yasinnn/menuju-indonesia-emas-2045/data/processed/fiscal_clustered_indonesia_2023.csv')

cluster_labels = {
    0: 'Dependent / Underperforming',
    1: 'Resource-Windfall Overperformers',
    2: 'Fiscally Autonomous',
    3: 'Stable / Average',
}
print('Loaded shape:', df.shape)

Loaded shape: (508, 22)


### 7.5 Geographic pattern

Do the clusters line up with real geography? We check the top provinces represented in each cluster.

In [6]:
for label in cluster_labels.values():
    subset = df[df['cluster_label'] == label]
    print(f"--- {label} (n={len(subset)}) ---")
    print(subset['provinsi'].value_counts().head(5).to_string())
    print()

--- Dependent / Underperforming (n=74) ---
provinsi
Nusa Tenggara Timur    9
Maluku                 8
Maluku Utara           6
Lampung                4
Papua Pegunungan       4

--- Resource-Windfall Overperformers (n=46) ---
provinsi
Kalimantan Selatan    8
Kalimantan Timur      7
Papua Tengah          5
Kalimantan Utara      4
Kalimantan Tengah     4

--- Fiscally Autonomous (n=156) ---
provinsi
Jawa Tengah    30
Jawa Timur     26
Jawa Barat     20
Bali            9
Banten          7

--- Stable / Average (n=232) ---
provinsi
Sumatera Utara       23
Aceh                 20
Sumatera Barat       15
Sulawesi Selatan     14
Sulawesi Tenggara    13





#####Reading the pattern reveals distinct, structurally entrenched fiscal behaviors across the archipelago (Siregar & Kurniawan, 2022):
* **Dependent / Underperforming** clusters concentrate heavily in NTT, Maluku, Maluku Utara, and Papua — mapping perfectly to the well-known eastern-Indonesia fiscal gap where local economies rely heavily on central balancing funds.

* **Resource-Windfall Overperformers** cluster across Kalimantan (Selatan/Timur/Tengah/Utara) and Papua Tengah — mining and resource royalty regions whose modest Own-Source Revenue (PAD) targets get blown past by massive commodity injections like sudden windfalls from coal, nickel, palm oil, or gold.

* **Fiscally Autonomous** clusters solidify across Jawa Tengah/Timur/Barat, Bali, and Banten — forming the nation's core industrial and tourism belt driven by mature internal consumer markets.

* **Stable / Average** spreads anchor across Sumatera and Sulawesi provinces — representing the largest, most "typical" macroeconomic baseline group in the distribution.

A caveat worth stating plainly in any writeup is that the silhouette scores were modest (~0.21), meaning these are soft, overlapping groupings rather than hard-edged, rigid categories. That is an accurate reflection of real-world public finance data where administrative boundaries blurred, not a modeling weakness to hide.


In [7]:
df.to_csv("fiscal_clustered_indonesia_2023.csv", index=False)
print("Final shape:", df.shape)

Final shape: (508, 22)


---
### 📤 Data Export & Handoff
* **Output File:** `fiscal_clustered_indonesia_2023.csv`
* **Next Destination:** `README.md` (results matrix / summary section)


## Summary

| Step | Outcome |
|---|---|
| Load & clean data | 508 regions, 8 fiscal indicators, no missing values |
| Handle skew | `log1p` transform applied, worked well for 6/8 indicators (2 flagged honestly) |
| Cluster (k=4) | Modest silhouette (~0.21) — soft, overlapping groups, not hard boundaries |
| Profile clusters | Dependent/Underperforming, Resource-Windfall Overperformers, Fiscally Autonomous, Stable/Average |
| Geographic check | Clusters align with known regional patterns (eastern Indonesia dependency, Java/Bali autonomy, Kalimantan resource windfalls) |

---


**Limitations** (deliberately out of scope):

spatial autocorrelation (needs region boundary geometry, not available), VaR/CVaR-style risk modeling (not statistically supportable on single-year cross-sectional ratios), transfer allocation optimization (inputs not in the current data).